TSV Generation Number 1

1. Using the patient tsv for each cell generate a range min to max removing outlier using the IQR median
2. Group the TSV based on the age range from  5-5.5, 5.5-6... all the way to to 21
3. Randomly generate new TSV based on the age ranges and tsvs, as well as numeric metadata

In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import os
#merged DF generator V1

# Path to the directory containing the TSV files
tsv_directory = "/content/drive/MyDrive/Data-Capstone/Second-Project/widsdatathon2025-university (1)/train_tsv/train_tsv"
metadata_file_path = "/content/drive/MyDrive/Data-Capstone/Second-Project/widsdatathon2025-university (1)/metadata/training_metadata.csv"  # Path to the newly uploaded metadata file

# STEP 1: Function to Extract Upper Triangular Matrix as 1D Vector
def extract_upper_triangle(file):
    matrix = pd.read_csv(file, sep='\t', header=None).values
    upper_tri_indices = np.triu_indices_from(matrix, k=1)
    return matrix[upper_tri_indices]

# STEP 2: Iterate Over All TSV Files in the Directory and Process Them
all_data = []  # List to store the processed data

# Iterate through all files in the specified directory
for filename in os.listdir(tsv_directory):
    if filename.endswith(".tsv"):  # Process only TSV files
        # Full file path
        file_path = os.path.join(tsv_directory, filename)

        # Extract upper triangle values from the TSV file
        upper_triangle_values = extract_upper_triangle(file_path)

        # Extract participant ID from the filename, removing "Copy of" if present
        participant_id = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")

        # Remove "Copy of" if it's part of the filename
        if participant_id.startswith("Copy of "):
            participant_id = participant_id.replace("Copy of ", "")

        # Convert upper triangle values into individual columns (each element in its own column)
        upper_triangle_columns = {f"upper_triangle_{i}": [value] for i, value in enumerate(upper_triangle_values)}

        # Add participant ID to the columns
        upper_triangle_columns["participant_id"] = [participant_id]

        # Create a DataFrame for this participant's data
        df = pd.DataFrame(upper_triangle_columns)

        # Reorder columns to ensure participant_id is the first column
        cols = ['participant_id'] + [col for col in df.columns if col != 'participant_id']
        df = df[cols]

        # Append to the list
        all_data.append(df)

# Concatenate all DataFrames into a single DataFrame
final_df = pd.concat(all_data, ignore_index=True)

# Step 3: Load the new metadata file
metadata_df = pd.read_csv(metadata_file_path)

# Step 4: Merge final_df with metadata_df on participant_id
merged_df = pd.merge(final_df, metadata_df, on="participant_id", how="left")

# Step 5: Reorder columns to shift metadata columns to the right
metadata_columns = [col for col in merged_df.columns if col not in final_df.columns]
final_columns = [col for col in merged_df.columns if col in final_df.columns]
final_columns += metadata_columns  # Metadata columns are shifted to the right
merged_df = merged_df[final_columns]

# Step 6: Save the final merged DataFrame to a CSV file
output_file_path = "/content/drive/MyDrive/Data-Capstone/Second-Project/Synthetic data/merged_data_with_metadata.csv"
merged_df.to_csv(output_file_path, index=False)

# Print a confirmation message with the file path
print(f"Data has been merged and written to {output_file_path}")


In [ ]:
import pandas as pd
import numpy as np
import os
# Split v1

# ——— 1) Paths — adjust as needed ———
merged_file = "/content/drive/MyDrive/Data-Capstone/Second-Project/Synthetic data/merged_data_with_metadata.csv"
output_dir  = "/content/drive/MyDrive/Data-Capstone/Second-Project/Synthetic data/filtered_data_groups"
os.makedirs(output_dir, exist_ok=True)

# ——— 2) IQR outlier filter ———
def filter_outliers(df):
    out = df.copy()
    for col in out.select_dtypes(include=[np.number]).columns:
        q1, q3 = out[col].quantile([0.25, 0.75])
        iqr    = q3 - q1
        out    = out[out[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]
    return out

# ——— 3) Load merged data ———
df = pd.read_csv(merged_file)

# ——— 4) Build dynamic 0.5-year bins ———
min_age = df['age'].min()
max_age = df['age'].max()
start   = np.floor(min_age * 2) / 2
stop    = np.ceil (max_age * 2) / 2 + 1e-6
edges   = np.arange(start, stop, 0.5)
bins    = [(edges[i], edges[i+1] - 0.01) for i in range(len(edges)-1)]

# ——— 5) Loop over bins, filter & save ———
for low, high in bins:
    grp = df[(df['age'] >= low) & (df['age'] <= high)]
    if grp.empty:
        continue

    # a) remove outliers
    filt = filter_outliers(grp)

    # b) save filtered CSV
    tag = f"{low:.2f}_{high:.2f}".replace('.', 'p')
    out_csv = os.path.join(output_dir, f"filtered_age_group_{tag}.csv")
    filt.to_csv(out_csv, index=False)

    # c) compute & save min/max summary
    summary = filt.describe().loc[['min','max']].T
    out_sum = os.path.join(output_dir, f"range_summary_age_group_{tag}.csv")
    summary.to_csv(out_sum)

    print(f"✔ Age {low:.2f}–{high:.2f}: {len(filt)} rows →")
    print(f"   • {out_csv}")
    print(f"   • {out_sum}")


In [ ]:
import pandas as pd
import numpy as np
import os
import random
#Random generator v1

# Paths (update as needed)
range_summary_directory = "/content/drive/MyDrive/Data-Capstone/Second-Project/Synthetic data/filtered_data_groups"
output_dir              = "/content/drive/MyDrive/Data-Capstone/Second-Project/Synthetic data/Generated CSV"

# Number of synthetic patients per group
num_samples_per_group = 65

# Define your age groups
age_groups = [
    (5.0, 5.49),
    (5.5, 5.99),
    (6.0, 6.49),
    (6.5, 6.99),
    (15.0, 15.49),
    (15.5, 15.99),
    (16.0, 16.49),
    (16.5, 16.99),
    (17.0, 17.49),
    (17.5, 17.99),
    (18.0, 18.49),
    (18.5, 18.99),
    (19.0, 19.49),
    (19.5, 19.99),
    (20.0, 20.49),
    (20.5, 20.99),
    (21.0, 21.49),
    (21.5, 21.99)
]

os.makedirs(output_dir, exist_ok=True)

def create_merged_dataset():
    for start_age, end_age in age_groups:
        # build the "13p00_13p49" style tag
        tag = f"{start_age:.2f}_{end_age:.2f}".replace('.', 'p')
        range_file = os.path.join(
            range_summary_directory,
            f"range_summary_age_group_{tag}.csv"
        )
        if not os.path.exists(range_file):
            print(f"Skipping {tag}: file not found")
            continue

        # load with feature names as index
        df_ranges = pd.read_csv(range_file, index_col=0)

        all_parts = []
        for _ in range(num_samples_per_group):
            part = {"participant_id": f"participant_{random.randint(1000,9999)}"}
            # age and bmi are also in the summary?
            for feat, row in df_ranges.iterrows():
                mn, mx = row["min"], row["max"]
                val = np.random.uniform(mn, mx) if mn < mx else mn
                if feat in ("age","bmi"):
                    part[feat] = round(val, 2)
                else:
                    part[feat] = val
            all_parts.append(part)

        out_path = os.path.join(output_dir, f"merged_synthetic_data_{tag}.csv")
        df_out = pd.DataFrame(all_parts)
        if os.path.exists(out_path):
            df_out.to_csv(out_path, mode='a', header=False, index=False)
            print(f"Appended {len(df_out)} to {out_path}")
        else:
            df_out.to_csv(out_path, index=False)
            print(f"Created {out_path}")

create_merged_dataset()


In [ ]:
import pandas as pd

# Replace with the path to your CSV file
csv_path = "/content/drive/MyDrive/Data-Capstone/Second-Project/Synthetic data/Generated CSV/merged_synthetic_data_17p00_17p49.csv"

# Load the CSV into a DataFrame
df = pd.read_csv(csv_path)

# Show the first five rows
print(df.head())
